In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
import time
from matplotlib.animation import FuncAnimation, PillowWriter
import ipywidgets as widgets
from IPython.display import display, HTML
import os # Import os for path handling

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Global variables to be controlled by widgets
GLOBAL_C_DIMENSIONAL = -10.0
GLOBAL_EPOCHS = 4000
GLOBAL_INITIALIZATION_TYPE = 'xavier_uniform' # 'xavier_uniform', 'kaiming_uniform', 'normal', 'zeros'
GLOBAL_IC_TYPE = 'exp(-10x^2)' # 'exp(-10x^2)', 'exp(-100x^2)'
GLOBAL_NUM_NEURONS = 128
GLOBAL_NUM_LAYERS = 3 # Excludes input/output layers, counts hidden layers
GLOBAL_PINN_TYPE = 'non-dimensional' # 'basic' or 'non-dimensional'

# Check for GPU
if torch.cuda.is_available():
    print("Using GPU!")
    device = torch.device("cuda")
else:
    print("Using CPU!")
    device = torch.device("cpu")

# --- 1. Define the Neural Network ---
class PINN(nn.Module):
    def __init__(self, num_neurons, num_layers, initialization_type):
        super(PINN, self).__init__()
        layers = []
        layers.append(nn.Linear(2, num_neurons)) # Input: x*, t
        layers.append(nn.Tanh())
        for _ in range(num_layers - 1): # Hidden layers
            layers.append(nn.Linear(num_neurons, num_neurons))
            layers.append(nn.Tanh())
        layers.append(nn.Linear(num_neurons, 1)) # Output: u*(x*,t)
        self.net = nn.Sequential(*layers)
        self.initialization_type = initialization_type
        self._init_weights()

    def _init_weights(self):
        for m in self.net:
            if isinstance(m, nn.Linear):
                if self.initialization_type == 'xavier_uniform':
                    nn.init.xavier_uniform_(m.weight)
                elif self.initialization_type == 'kaiming_uniform':
                    nn.init.kaiming_uniform_(m.weight, nonlinearity='tanh') # Kaiming is for ReLU, but can be adapted
                elif self.initialization_type == 'normal':
                    nn.init.normal_(m.weight, mean=0.0, std=0.01)
                elif self.initialization_type == 'zeros':
                    nn.init.zeros_(m.weight)
                
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x)

# --- 2. Define the Loss Functions ---
def pde_loss_fn(model, x_pde, t_pde, pde_constant):
    x_pde.requires_grad_(True)
    t_pde.requires_grad_(True)

    inputs = torch.cat([x_pde, t_pde], dim=1)
    u_star = model(inputs)

    u_star_t = torch.autograd.grad(u_star, t_pde, grad_outputs=torch.ones_like(u_star), create_graph=True)[0]
    u_star_x = torch.autograd.grad(u_star, x_pde, grad_outputs=torch.ones_like(u_star), create_graph=True)[0]

    pde_residual = u_star_t + pde_constant * u_star_x
    return torch.mean(pde_residual**2)

def ic_loss_fn(model, x_ic, t_ic, K_IC_ND_val):
    inputs_ic = torch.cat([x_ic, t_ic], dim=1)
    u_pred_ic = model(inputs_ic)
    u_target_ic = torch.exp(-K_IC_ND_val * x_ic**2)
    return torch.mean((u_pred_ic - u_target_ic)**2)

# --- 3. Generate Collocation Points ---
def generate_collocation_points(N_pde, N_ic, domain_x_nd, domain_t_dim, device):
    x_pde = torch.rand(N_pde, 1, device=device) * (domain_x_nd[1] - domain_x_nd[0]) + domain_x_nd[0]
    t_pde = torch.rand(N_pde, 1, device=device) * (domain_t_dim[1] - domain_t_dim[0]) + domain_t_dim[0]

    x_ic = torch.rand(N_ic, 1, device=device) * (domain_x_nd[1] - domain_x_nd[0]) + domain_x_nd[0]
    t_ic = torch.full_like(x_ic, domain_t_dim[0])

    return x_pde, t_pde, x_ic, t_ic

# --- Main Training Loop ---
def train_pinn_interactive(
    c_dimensional_val,
    epochs_val,
    initialization_type_val,
    ic_type_val,
    num_neurons_val,
    num_layers_val,
    pinn_type_val
):
    # Update global parameters based on widget values
    global GLOBAL_C_DIMENSIONAL, GLOBAL_EPOCHS, GLOBAL_INITIALIZATION_TYPE, GLOBAL_IC_TYPE, \
           GLOBAL_NUM_NEURONS, GLOBAL_NUM_LAYERS, GLOBAL_PINN_TYPE

    GLOBAL_C_DIMENSIONAL = c_dimensional_val
    GLOBAL_EPOCHS = epochs_val
    GLOBAL_INITIALIZATION_TYPE = initialization_type_val
    GLOBAL_IC_TYPE = ic_type_val
    GLOBAL_NUM_NEURONS = num_neurons_val
    GLOBAL_NUM_LAYERS = num_layers_val
    GLOBAL_PINN_TYPE = pinn_type_val

    # Configuration Parameters (adjusted based on PINN_TYPE)
    U_CHAR_DIMENSIONAL = 10.0 # Characteristic displacement
    L_CHAR_DIMENSIONAL = 10.0 # Characteristic length

    if GLOBAL_PINN_TYPE == 'non-dimensional':
        PDE_CONSTANT_FIRST_ORDER = GLOBAL_C_DIMENSIONAL / L_CHAR_DIMENSIONAL
        DOMAIN_X_ND = [-1.0, 1.0]
        DOMAIN_T_DIMENSIONAL = [0.0, 1.0]
        if GLOBAL_IC_TYPE == 'exp(-10x^2)':
            K_IC_ND = 10.0 * L_CHAR_DIMENSIONAL**2
        else: # exp(-100x^2)
            K_IC_ND = 100.0 * L_CHAR_DIMENSIONAL**2
        plot_amplitude_factor = U_CHAR_DIMENSIONAL # For plotting the analytical solution
        analytical_ic_original_k = 10.0 if GLOBAL_IC_TYPE == 'exp(-10x^2)' else 100.0
    else: # Basic PINN (dimensional)
        PDE_CONSTANT_FIRST_ORDER = GLOBAL_C_DIMENSIONAL
        DOMAIN_X_ND = [-10.0, 10.0] # Use dimensional domain for basic
        DOMAIN_T_DIMENSIONAL = [0.0, 1.0]
        K_IC_ND = 10.0 if GLOBAL_IC_TYPE == 'exp(-10x^2)' else 100.0 # K_IC directly for dimensional form
        plot_amplitude_factor = 10.0 # Initial amplitude is 10 for both IC types
        analytical_ic_original_k = K_IC_ND

    N_PDE_POINTS = 20000
    N_IC_POINTS = 1000
    LEARNING_RATE = 1e-3
    W_PDE = 1.0
    W_IC = 1.0

    print(f"\n--- Training {GLOBAL_PINN_TYPE.upper()} PINN with parameters ---")
    print(f"C: {GLOBAL_C_DIMENSIONAL}")
    print(f"Epochs: {GLOBAL_EPOCHS}")
    print(f"Initialization: {GLOBAL_INITIALIZATION_TYPE}")
    print(f"Initial Condition: {GLOBAL_IC_TYPE}")
    print(f"Neurons per Layer: {GLOBAL_NUM_NEURONS}")
    print(f"Number of Hidden Layers: {GLOBAL_NUM_LAYERS}")
    print("-" * 40)

    model = PINN(GLOBAL_NUM_NEURONS, GLOBAL_NUM_LAYERS, GLOBAL_INITIALIZATION_TYPE).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    x_pde, t_pde, x_ic, t_ic = generate_collocation_points(
        N_PDE_POINTS, N_IC_POINTS, DOMAIN_X_ND, DOMAIN_T_DIMENSIONAL, device
    )

    start_time = time.time()
    for epoch in range(GLOBAL_EPOCHS):
        model.train()
        optimizer.zero_grad()

        loss_pde = pde_loss_fn(model, x_pde, t_pde, PDE_CONSTANT_FIRST_ORDER)
        loss_ic = ic_loss_fn(model, x_ic, t_ic, K_IC_ND)

        total_loss = W_PDE * loss_pde + W_IC * loss_ic
        total_loss.backward()
        optimizer.step()

        if (epoch + 1) % 1000 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}/{GLOBAL_EPOCHS}, Total Loss: {total_loss.item():.6f}, "
                  f"PDE Loss: {loss_pde.item():.6f}, IC Loss: {loss_ic.item():.6f}")

    end_time = time.time()
    print(f"\nTraining finished in {end_time - start_time:.2f} seconds.")

    # --- Analytical Solution Function (for Dimensional Space) ---
    def analytical_solution_dimensional(x, t, C_dim, K_IC_original, amplitude_factor):
        return amplitude_factor * np.exp(-K_IC_original * (x - C_dim * t)**2)

    # --- 4. Visualize Results (Static Plots in Actual Space) ---
    def plot_static_results_interactive(model):
        model.eval()

        x_test_dim = np.linspace(-10.0, 10.0, 200)
        t_test_dim = np.linspace(0.0, 1.0, 100)
        X_DIM, T_DIM = np.meshgrid(x_test_dim, t_test_dim)

        # Prepare inputs for the model based on PINN_TYPE
        if GLOBAL_PINN_TYPE == 'non-dimensional':
            X_model_input = X_DIM / L_CHAR_DIMENSIONAL
            T_model_input = T_DIM
            output_scaling_factor = U_CHAR_DIMENSIONAL
        else: # Basic PINN (dimensional)
            X_model_input = X_DIM
            T_model_input = T_DIM
            output_scaling_factor = 1.0 # Model directly outputs dimensional u

        x_flat_model_input = torch.tensor(X_model_input.flatten().reshape(-1, 1), dtype=torch.float32).to(device)
        t_flat_model_input = torch.tensor(T_model_input.flatten().reshape(-1, 1), dtype=torch.float32).to(device)
        inputs_flat = torch.cat([x_flat_model_input, t_flat_model_input], dim=1)

        with torch.no_grad():
            u_pred_flat = model(inputs_flat).cpu().numpy()
        U_pred_reshaped = u_pred_flat.reshape(X_DIM.shape)
        U_pred_dim = U_pred_reshaped * output_scaling_factor

        U_analytical_dim = analytical_solution_dimensional(X_DIM, T_DIM, GLOBAL_C_DIMENSIONAL, analytical_ic_original_k, plot_amplitude_factor)

        # Plot 3D surface of PINN solution
        fig1 = plt.figure(figsize=(12, 8))
        ax1 = fig1.add_subplot(111, projection='3d')
        ax1.plot_surface(X_DIM, T_DIM, U_pred_dim, cmap=cm.viridis, antialiased=False)
        ax1.set_xlabel('x (actual)')
        ax1.set_ylabel('t (actual)')
        ax1.set_zlabel('u(x,t) (actual)')
        ax1.set_title(f'{GLOBAL_PINN_TYPE.capitalize()} PINN Solution (C={GLOBAL_C_DIMENSIONAL}, IC Amp={plot_amplitude_factor})')
        plt.tight_layout()
        plt.show()

        # Plot 3D surface of Analytical solution
        fig2 = plt.figure(figsize=(12, 8))
        ax2 = fig2.add_subplot(111, projection='3d')
        ax2.plot_surface(X_DIM, T_DIM, U_analytical_dim, cmap=cm.viridis, antialiased=False)
        ax2.set_xlabel('x (actual)')
        ax2.set_ylabel('t (actual)')
        ax2.set_zlabel('u(x,t) (actual)')
        ax2.set_title(f'Analytical Solution (C={GLOBAL_C_DIMENSIONAL}, IC Amp={plot_amplitude_factor})')
        plt.tight_layout()
        plt.show()

        # Plot difference (Error)
        fig3 = plt.figure(figsize=(12, 8))
        ax3 = fig3.add_subplot(111, projection='3d')
        diff = np.abs(U_pred_dim - U_analytical_dim)
        surf = ax3.plot_surface(X_DIM, T_DIM, diff, cmap=cm.plasma, antialiased=False)
        fig3.colorbar(surf, shrink=0.5, aspect=5, label='Absolute Error')
        ax3.set_xlabel('x (actual)')
        ax3.set_ylabel('t (actual)')
        ax3.set_zlabel('Absolute Error')
        ax3.set_title('Absolute Error |PINN - Analytical|')
        plt.tight_layout()
        plt.show()

        # Plot snapshots at different actual times
        fig4, axes = plt.subplots(1, 3, figsize=(18, 5))
        times_to_plot_dim = [0.0, 0.25, 0.5]

        for i, t_snap_dim in enumerate(times_to_plot_dim):
            ax = axes[i]
            if GLOBAL_PINN_TYPE == 'non-dimensional':
                x_test_model_input = torch.tensor(x_test_dim / L_CHAR_DIMENSIONAL, dtype=torch.float32).reshape(-1, 1).to(device)
                t_snap_tensor_model_input = torch.full((len(x_test_dim), 1), t_snap_dim, device=device, dtype=torch.float32)
                
                inputs_snap = torch.cat([x_test_model_input, t_snap_tensor_model_input], dim=1)
                with torch.no_grad():
                    u_pred_snap_star = model(inputs_snap).cpu().numpy().flatten()
                u_snap_pred_dim = u_pred_snap_star * U_CHAR_DIMENSIONAL
            else: # Basic PINN (dimensional)
                x_test_model_input = torch.tensor(x_test_dim, dtype=torch.float32).reshape(-1, 1).to(device)
                t_snap_tensor_model_input = torch.full((len(x_test_dim), 1), t_snap_dim, device=device, dtype=torch.float32)

                inputs_snap = torch.cat([x_test_model_input, t_snap_tensor_model_input], dim=1)
                with torch.no_grad():
                    u_snap_pred_dim = model(inputs_snap).cpu().numpy().flatten() # Direct output

            ax.plot(x_test_dim, u_snap_pred_dim, label='PINN Prediction', color='blue')
            u_snap_analytical_dim = analytical_solution_dimensional(x_test_dim, t_snap_dim, GLOBAL_C_DIMENSIONAL, analytical_ic_original_k, plot_amplitude_factor)
            ax.plot(x_test_dim, u_snap_analytical_dim, '--', label='Analytical Solution', color='red', alpha=0.7)

            ax.set_title(f't = {t_snap_dim:.2f}')
            ax.set_xlabel('x (actual)')
            ax.set_ylabel('u(x,t) (actual)')
            ax.grid(True)
            ax.legend()
            ax.set_ylim([-1.0, 11.0])

        plt.tight_layout()
        plt.show()

    plot_static_results_interactive(model)

# --- 5. Create Animation ---
def create_animation_interactive(model):
    model.eval()

    x_anim_dim = np.linspace(-10.0, 10.0, 300)
    t_frames_dim = np.linspace(0.0, 1.0, 100) # 100 frames for the animation

    fig, ax = plt.subplots(figsize=(10, 6))
    line_pinn, = ax.plot([], [], 'b-', label='PINN Prediction')
    line_analytical, = ax.plot([], [], 'r--', label='Analytical Solution', alpha=0.7)
    title = ax.set_title('')
    ax.set_xlabel('x (actual)')
    ax.set_ylabel('u(x,t) (actual)')
    ax.set_xlim([-10.0, 10.0])
    ax.set_ylim([-1.0, 11.0])
    ax.legend()
    ax.grid(True)

    animation_interval_ms = 100 # Milliseconds between frames (100ms = 10 fps)
    fps_val = int(1000 / animation_interval_ms)

    def animate(i):
        current_t_dim = t_frames_dim[i]

        if GLOBAL_PINN_TYPE == 'non-dimensional':
            x_anim_model_input = torch.tensor(x_anim_dim / L_CHAR_DIMENSIONAL, dtype=torch.float32).reshape(-1, 1).to(device)
            t_tensor_model_input = torch.full((len(x_anim_dim), 1), current_t_dim, device=device, dtype=torch.float32)
            inputs = torch.cat([x_anim_model_input, t_tensor_model_input], dim=1)

            with torch.no_grad():
                u_star_pinn = model(inputs).cpu().numpy().flatten()
            u_pinn_dim = u_star_pinn * U_CHAR_DIMENSIONAL
        else: # Basic PINN (dimensional)
            x_anim_model_input = torch.tensor(x_anim_dim, dtype=torch.float32).reshape(-1, 1).to(device)
            t_tensor_model_input = torch.full((len(x_anim_dim), 1), current_t_dim, device=device, dtype=torch.float32)
            inputs = torch.cat([x_anim_model_input, t_tensor_model_input], dim=1)
            
            with torch.no_grad():
                u_pinn_dim = model(inputs).cpu().numpy().flatten() # Direct output

        u_analytical_dim = analytical_solution_dimensional(x_anim_dim, current_t_dim, GLOBAL_C_DIMENSIONAL, analytical_ic_original_k, plot_amplitude_factor)

        line_pinn.set_data(x_anim_dim, u_pinn_dim)
        line_analytical.set_data(x_anim_dim, u_analytical_dim)
        title.set_text(f'Wave Propagation ({GLOBAL_PINN_TYPE.capitalize()} PINN) at t = {current_t_dim:.3f}')
        return line_pinn, line_analytical, title

    print("Creating animation (this may take a few minutes)...")

    # Define a clear filename for the GIF
    gif_filename = 'pinn_wave_simulation.gif'

    try:
        # Create the animation object
        anim = FuncAnimation(fig, animate, frames=len(t_frames_dim), interval=animation_interval_ms, blit=False)

        # Save the animation to a file
        
        anim.save(gif_filename, writer='pillow', fps=fps_val)
        print(f"Animation saved to: {os.path.abspath(gif_filename)}") # Print absolute path for clarity
        
        plt.close(fig) # Close the figure to prevent static display

        # Display the saved GIF in the Jupyter output
        display(HTML(f'<img src="{gif_filename}" alt="PINN Wave Animation">'))
        print("Animation displayed above.")

    except Exception as e:
        print(f"\nERROR: Could not create or display animation.")
        print(f"Please ensure you have 'Pillow' installed (`pip install Pillow`).")
        print(f"Also, check if your Matplotlib version and Jupyter environment are compatible.")
        print(f"Detailed error: {e}")
        # As a last resort, if file saving fails, you might manually generate the JSHTML.
        # This is often very large and can crash browsers for complex animations.
        # print("Attempting to display using anim.to_jshtml() as a fallback (may be slow/heavy)...")
        # try:
        #     display(HTML(anim.to_jshtml()))
        # except Exception as js_e:
        #     print(f"Failed with to_jshtml() fallback: {js_e}")


# --- IPython Widgets Setup ---

# C (wave speed)
c_widget = widgets.FloatText(
    value=GLOBAL_C_DIMENSIONAL,
    description='C:',
    disabled=False
)

# Number of Epochs
epochs_widget = widgets.IntText(
    value=GLOBAL_EPOCHS,
    description='Epochs:',
    disabled=False
)

# Initialization Types
initialization_widget = widgets.Dropdown(
    options=['xavier_uniform', 'kaiming_uniform', 'normal', 'zeros'],
    value=GLOBAL_INITIALIZATION_TYPE,
    description='Init Type:',
    disabled=False
)

# Initial Conditions (ICs)
ic_widget = widgets.Dropdown(
    options=['exp(-10x^2)', 'exp(-100x^2)'],
    value=GLOBAL_IC_TYPE,
    description='IC Type:',
    disabled=False
)

# Number of Neurons
neurons_widget = widgets.IntText(
    value=GLOBAL_NUM_NEURONS,
    description='Neurons:',
    disabled=False
)

# Number of Hidden Layers
layers_widget = widgets.IntText(
    value=GLOBAL_NUM_LAYERS,
    description='Layers:',
    disabled=False
)

# PINN Type (Basic vs. Non-Dimensional)
pinn_type_widget = widgets.RadioButtons(
    options=['non-dimensional', 'basic'],
    value=GLOBAL_PINN_TYPE,
    description='PINN Type:',
    disabled=False
)

# Button to trigger training
train_button = widgets.Button(description="Train PINN and Visualize")
output_area = widgets.Output()

def on_train_button_clicked(b):
    with output_area:
        output_area.clear_output() # Clear previous output
        train_pinn_interactive(
            c_widget.value,
            epochs_widget.value,
            initialization_widget.value,
            ic_widget.value,
            neurons_widget.value,
            layers_widget.value,
            pinn_type_widget.value
        )

train_button.on_click(on_train_button_clicked)

# Arrange widgets in a VBox
ui = widgets.VBox([
    c_widget,
    epochs_widget,
    initialization_widget,
    ic_widget,
    neurons_widget,
    layers_widget,
    pinn_type_widget,
    train_button,
    output_area
])

display(ui)